# Road zone detection - NVIDIA LocateAnything-3B

One call per source video (camera is stationary for the full clip - confirmed via telemetry
- so the road's screen position never changes), not per frame. Output (the zone box, in
both pixel and normalized coordinates) is saved to a small JSON file per source so this
model never needs to be called again - future SAHI runs just load the saved zone.

4-bit quantized proactively: NVIDIA's own numbers show ~35GB peak memory with dense
attention vs ~11.7GB with FlashAttention, and Kaggle's T4 (Turing arch) doesn't properly
support the FlashAttention backend that gets it down to 11.7GB - same OOM risk pattern hit
earlier tonight with Qwen3-VL on this same GPU class.

Not gated (unlike SAM3/3.1, which need manual HF approval) - confirmed earlier via
huggingface_hub.HfApi().model_info(), not assumed.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
!pip install -q transformers==4.57.1 accelerate bitsandbytes huggingface_hub decord==0.6.0 lmdb==1.7.5

In [ ]:
import glob
import cv2
from PIL import Image

SOURCES = ["multiroad", "intersection"]
frames = {}
for source in SOURCES:
    found = glob.glob(f"/kaggle/input/**/{source}_10s.MP4", recursive=True)
    assert found, f"No {source}_10s.MP4 found under /kaggle/input"
    cap = cv2.VideoCapture(found[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, 150)  # mid-clip frame, ~5s in
    ok, frame = cap.read()
    cap.release()
    assert ok, f"Failed to read frame from {found[0]}"
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frames[source] = image
    print(source, "->", found[0], image.size)

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoProcessor, BitsAndBytesConfig

model_path = "nvidia/LocateAnything-3B"
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_path, quantization_config=quant_config, device_map="cuda:0", trust_remote_code=True,
)
print("Loaded OK")

In [ ]:
import re
import json
from PIL import ImageDraw

PROMPT = (
    "Locate the region that matches the following description: the paved road surface "
    "where vehicles drive."
)

def parse_boxes(text, w, h):
    boxes = []
    for m in re.finditer(r"<box><(\d+)><(\d+)><(\d+)><(\d+)></box>", text):
        x1, y1, x2, y2 = [int(g) for g in m.groups()]
        boxes.append({
            "pixel": [x1 / 1000 * w, y1 / 1000 * h, x2 / 1000 * w, y2 / 1000 * h],
            "normalized_0_1000": [x1, y1, x2, y2],
        })
    return boxes

OUTPUT_DIR = "/kaggle/working/road_zones"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = {}
for source, image in frames.items():
    print(f"\n=== {source} ===")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    generated = model.generate(
        **inputs, tokenizer=tokenizer, max_new_tokens=512, do_sample=False, use_cache=True,
    )
    answer = processor.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    print("Raw answer:", answer)

    boxes = parse_boxes(answer, *image.size)
    print(f"Parsed {len(boxes)} box(es)")

    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    for b in boxes:
        draw.rectangle(b["pixel"], outline="red", width=6)
    preview_path = f"{OUTPUT_DIR}/{source}_zone_preview.jpg"
    annotated.save(preview_path)

    results[source] = {
        "raw_answer": answer,
        "boxes": boxes,
        "image_size": list(image.size),
        "preview": preview_path,
    }
    print("Saved preview:", preview_path)

with open(f"{OUTPUT_DIR}/road_zones.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved combined result:", f"{OUTPUT_DIR}/road_zones.json")

## Next

Pull `/kaggle/working/road_zones` and look at both `*_zone_preview.jpg` files before trusting
this for anything - visual review first, per the plan. If good: `road_zones.json` is the
cached result to reuse in future SAHI runs (restrict tiling to this box, or its slice-
intersecting region) - no need to call this model again. If not sane: fall back to manual
ROI selection instead of iterating on prompts.